In [30]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv("gs://ukbb_spleen/CHIPCAD_pheno_v5.csv", sep='\s+')
mri_data = pd.read_csv("gs://ukbb_spleen/CAD_Phenotypes_for_MRI_Participants.csv")

In [21]:
# open the radiomics results
rad_2 = pd.read_csv("../radiomics/radiomics_spleen_1.csv")
rad_3 = pd.read_csv("../radiomics/radiomics_spleen3_1.csv")

phase, organ = 'wat', 'spleen'
diag_cols = [elem for elem in rad_2.columns if 'diagnostics' in elem]
rad_2 = rad_2[rad_2.Image.str.contains(phase)]
rad_2 = rad_2.drop(['Image', 'Mask']+diag_cols, axis = 1)
rad_2 = rad_2.add_prefix(organ+'_')
rad_2 = rad_2.rename(columns = {organ+'_ID':'ID'})

diag_cols = [elem for elem in rad_3.columns if 'diagnostics' in elem]
rad_3 = rad_3[rad_3.Image.str.contains(phase)]
rad_3 = rad_3.drop(['Image', 'Mask']+diag_cols, axis = 1)
rad_3 = rad_3.add_prefix(organ+'_')
rad_3 = rad_3.rename(columns = {organ+'_ID':'ID'})


In [22]:
print(rad_2.shape)
print(rad_3.shape)
rad_3.columns = [str(col) + '_3' for col in rad_3.columns]
rad_3 = rad_3.rename(columns = {'ID_3':'ID'})

merged = pd.merge(rad_2, rad_3, on = "ID", how = "inner")
print(merged.shape)
print(len(merged.ID.unique()))

(43198, 109)
(5196, 109)
(1045, 217)
1033


In [23]:
merged.head()

,spleen_Unnamed: 0,ID,spleen_original_shape_Elongation,spleen_original_shape_Flatness,spleen_original_shape_LeastAxisLength,spleen_original_shape_MajorAxisLength,spleen_original_shape_Maximum2DDiameterColumn,spleen_original_shape_Maximum2DDiameterRow,spleen_original_shape_Maximum2DDiameterSlice,spleen_original_shape_Maximum3DDiameter,...,spleen_original_glszm_SmallAreaHighGrayLevelEmphasis_3,spleen_original_glszm_SmallAreaLowGrayLevelEmphasis_3,spleen_original_glszm_ZoneEntropy_3,spleen_original_glszm_ZonePercentage_3,spleen_original_glszm_ZoneVariance_3,spleen_original_ngtdm_Busyness_3,spleen_original_ngtdm_Coarseness_3,spleen_original_ngtdm_Complexity_3,spleen_original_ngtdm_Contrast_3,spleen_original_ngtdm_Strength_3
0,150wat.nii.gz,1988214,0.681635,0.385479,15.031059,38.99317,41.231056,41.340053,31.240999,42.591079,...,31.113297,0.036174,5.666778,0.038344,56917.924485,4.921251,0.001032,64.677593,0.010752,0.134485
1,140wat.nii.gz,4655396,0.735682,0.398786,14.62511,36.67412,41.231056,38.418745,33.286634,41.533119,...,43.701761,0.015368,6.004631,0.050401,21571.456293,3.468658,0.001193,111.713312,0.024084,0.148538
2,139wat.nii.gz,1171878,0.684792,0.391569,19.012592,48.554935,52.201533,45.221676,42.154478,53.197744,...,47.113389,0.019934,5.937912,0.033967,67972.280054,6.68754,0.000711,87.590007,0.016525,0.104224
3,135wat.nii.gz,3694192,0.627984,0.323777,10.908028,33.689976,35.227830,34.234486,26.248809,36.069378,...,18.734214,0.056029,5.419562,0.061728,9847.274754,3.384114,0.002221,75.277229,0.027695,0.187722
4,124wat.nii.gz,3507596,0.617597,0.413029,15.705868,38.026066,36.878178,39.408121,28.017851,42.638011,...,43.202597,0.016385,5.798457,0.04448,34382.944746,3.908703,0.00108,95.388902,0.017171,0.161988


In [26]:
mri_data.columns

Index(['f.eid', 'MRI_Date', 'filename', 'composite_mi_cad_stroke_Date',
       'composite_mi_cad_stroke', 'Coronary_Artery_Disease_Date',
       'Coronary_Artery_Disease', 'Coronary_Artery_Disease_HARD_Date',
       'Coronary_Artery_Disease_HARD',
       'Coronary_Artery_Disease_INTERMEDIATE_Date',
       'Coronary_Artery_Disease_INTERMEDIATE',
       'Coronary_Artery_Disease_SOFT_Date', 'Coronary_Artery_Disease_SOFT',
       'Ischemic_stroke', 'Ischemic_stroke_Date', 'Myocardial_Infarction',
       'Myocardial_Infarction_Date', 'Date_of_Death', 'Last_Follow_Up',
       'time_to_follow_up', 'Years_To_composite_mi_cad_stroke',
       'Prevalent_composite_mi_cad_stroke', 'Incident_composite_mi_cad_stroke',
       'Years_To_Coronary_Artery_Disease', 'Prevalent_Coronary_Artery_Disease',
       'Incident_Coronary_Artery_Disease',
       'Years_To_Coronary_Artery_Disease_HARD',
       'Prevalent_Coronary_Artery_Disease_HARD',
       'Incident_Coronary_Artery_Disease_HARD',
       'Years_To_C

In [64]:
# filtere to people whose CAD date is after the mri visit 2 and before mri visit 3
mri_data_filt = mri_data[['MRI_Date', 'filename', 'Coronary_Artery_Disease_SOFT_Date', 'f.eid']]
mri_data_v2 = mri_data_filt[mri_data_filt['filename'].str.contains('_2_0.zip')]
print(mri_data_v2.shape)
#mri_data_v2 = mri_data_v2[mri_data_v2['MRI_Date']<mri_data_v2['Coronary_Artery_Disease_SOFT_Date']]
print(mri_data_v2.shape)
mri_data_v3 = mri_data_filt[mri_data_filt['filename'].str.contains('_3_0.zip')]
print(mri_data_v3.shape)
#mri_data_v3 = mri_data_v3[mri_data_v3['MRI_Date']>mri_data_v3['Coronary_Artery_Disease_SOFT_Date']]
print(mri_data_v3.shape)

common_IDs = list(set(mri_data_v2['f.eid'].unique()) & set(mri_data_v3['f.eid'].unique()))

(43404, 4)
(43404, 4)
(2556, 4)
(2556, 4)


In [65]:
len(common_IDs)

2429

In [35]:
mri_data_filt.MRI_Date_3.value_counts(dropna=True)

2019-09-20    24
2019-09-02    22
2019-11-18    21
2019-10-05    21
2020-01-07    20
              ..
2019-06-02     1
2019-05-22     1
2019-06-08     1
2019-06-06     1
2019-05-29     1
Name: MRI_Date_3, Length: 239, dtype: int64

In [38]:
res = mri_data_filt.dropna(subset=['MRI_Date_3'])
res = res.dropna(subset=['MRI_Date_2'])
res.shape

(0, 6)